# Part 1: Code Q&A System with Bash Tools

This notebook implements a RAG system that answers questions about the `mcp-gateway-registry` codebase using **bash tools** (grep, find, cat, tree) instead of traditional vector retrieval.

## System Architecture
```
┌─────────────┐     ┌──────────────────┐     ┌─────────────────┐     ┌─────────────┐
│ User Query  │────>│  Query Router    │────>│  Bash Tool(s)   │────>│ LLM Answer  │
│             │     │ (classify query) │     │  Execute & Get  │     │ with Context│
└─────────────┘     └──────────────────┘     │  File Contents  │     └─────────────┘
                                             └─────────────────┘
```

## Steps
1. **Classify** the query type
2. **Select** appropriate bash tools
3. **Execute** commands and collect output
4. **Pass** context + question to LLM
5. **Generate** answer with file references

## Step 0: Setup — Clone the Repository
Run this once to clone the target codebase into your project directory.

In [2]:
# Clone the mcp-gateway-registry codebase (run only once)
import os

if not os.path.exists("./mcp-gateway-registry"):
    os.system("git clone https://github.com/agentic-community/mcp-gateway-registry")
    print("Repository cloned successfully.")
else:
    print("Repository already exists, skipping clone.")

Cloning into 'mcp-gateway-registry'...


Repository cloned successfully.


## Step 1: Imports & Configuration

In [1]:
import os
import json
import subprocess
from groq import Groq

# ── Configuration ──────────────────────────────────────────────────────────────
REPO_PATH = "./mcp-gateway-registry"   # path to the cloned codebase
MAX_CONTEXT_CHARS = 12000              # max characters passed to LLM to avoid token overflow
GROQ_MODEL = "llama-3.3-70b-versatile" # Groq model to use

# Load API key from environment variable (set via: export GROQ_API_KEY="gsk_...")
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found. Please run: export GROQ_API_KEY='your_key'")

client = Groq(api_key=api_key)
print("Configuration loaded successfully.")
print(f"  Model : {GROQ_MODEL}")
print(f"  Repo  : {REPO_PATH}")
print(f"  Max context chars: {MAX_CONTEXT_CHARS}")

Configuration loaded successfully.
  Model : llama-3.3-70b-versatile
  Repo  : ./mcp-gateway-registry
  Max context chars: 12000


## Step 2: Module 1 — Bash Tool Executor

This module safely executes bash commands and returns their output as a string.
It handles timeouts and automatically truncates output that is too long.

In [3]:
def execute_bash(command: str, max_chars: int = MAX_CONTEXT_CHARS) -> str:
    """
    Safely execute a bash command and return its stdout as a string.
    
    Args:
        command:   The bash command string to execute.
        max_chars: Maximum characters to return (truncates if exceeded).
    
    Returns:
        Command output as a string, or an error/timeout message.
    """
    try:
        result = subprocess.run(
            command,
            shell=True,
            capture_output=True,
            text=True,
            timeout=30
        )
        output = result.stdout

        # Fall back to stderr if stdout is empty (e.g., some find/grep warnings)
        if result.stderr and not output:
            output = result.stderr

        # Truncate to avoid exceeding LLM context window
        if len(output) > max_chars:
            output = output[:max_chars] + f"\n\n... [truncated — total {len(result.stdout)} chars]"

        return output.strip() if output.strip() else "(no output)"

    except subprocess.TimeoutExpired:
        return "(command timed out after 30s)"
    except Exception as e:
        return f"(execution error: {e})"


# ── Quick sanity check ─────────────────────────────────────────────────────────
print("Testing bash executor...")
test_output = execute_bash(f"ls {REPO_PATH} | head -10")
print(test_output)

Testing bash executor...
CLAUDE.md
CODE_OF_CONDUCT.md
CONTRIBUTING.md
DEV_INSTRUCTIONS.md
Dockerfile
LICENSE
Makefile
NOTICE
README.md
SECURITY.md


## Step 3: Module 2 — Query Router

The router classifies the user question and decides which bash commands to run.
It uses the LLM itself to reason about the best retrieval strategy.

| Query Type | Example | Bash Tools |
|---|---|---|
| `dependency` | "What Python packages does this use?" | `cat pyproject.toml`, `cat package.json` |
| `structure` | "What languages are used?" | `tree`, `find` |
| `code_search` | "How does authentication work?" | `grep` + `cat` |
| `documentation` | "How do I deploy?" | `find docs/`, `cat` |
| `multi` | Complex multi-file questions | combination of above |

In [4]:
def classify_query(question: str) -> dict:
    """
    Use the LLM to classify the question and generate bash commands for retrieval.
    
    Args:
        question: The user's natural language question about the codebase.
    
    Returns:
        A dict with keys: query_type, reasoning, commands (list of bash strings).
    """
    prompt = f"""You are a codebase query router. Given a user question, decide which bash commands
to run to retrieve the most relevant information from the repository.

Repository path: {REPO_PATH}

Available tools:
- tree: show directory structure
- find: locate files by name or extension
- cat: read file contents
- grep: search for patterns inside files

Query type hints:
- dependency     → cat pyproject.toml, cat package.json
- structure      → tree -L 3, find with extension filters
- code_search    → grep for keywords + cat relevant files
- documentation  → find docs/ + cat markdown files
- multi          → combine grep, find, cat across code and docs

User question: {question}

Return ONLY valid JSON (no markdown, no extra text):
{{
    "query_type": "dependency | structure | code_search | documentation | multi",
    "reasoning": "one sentence explaining the chosen strategy",
    "commands": [
        "bash command 1",
        "bash command 2"
    ]
}}

Rules:
- Maximum 5 commands
- Each command must be directly executable in a terminal
- Use --include filters with grep to target specific file types
- For complex questions, first grep to find relevant files, then cat those files
"""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,   # low temperature for deterministic routing
        max_tokens=800
    )

    raw = response.choices[0].message.content.strip()

    # Strip markdown code fences if the LLM wraps output in ```json ... ```
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fallback strategy if JSON parsing fails
        print(f"[Warning] Router JSON parse failed, using fallback strategy.\nRaw output: {raw}")
        return {
            "query_type": "multi",
            "reasoning": "JSON parse failed, using fallback",
            "commands": [
                f"tree {REPO_PATH} -L 3 2>/dev/null",
                f"grep -r '{question[:40]}' {REPO_PATH} --include='*.py' --include='*.md' -l 2>/dev/null | head -10"
            ]
        }


# ── Quick test: route a simple question ───────────────────────────────────────
print("Testing query router...")
test_classification = classify_query("What Python dependencies does this project use?")
print(json.dumps(test_classification, indent=2))

Testing query router...
{
  "query_type": "dependency",
  "reasoning": "The user is asking about Python dependencies, so we need to examine the project's configuration files to find this information.",
  "commands": [
    "cat ./mcp-gateway-registry/pyproject.toml"
  ]
}


## Step 4: Module 3 — Context Retriever

Executes all bash commands returned by the router and assembles the context string.

In [5]:
def retrieve_context(classification: dict) -> str:
    """
    Execute all bash commands from the router and collect their outputs.
    
    Args:
        classification: The dict returned by classify_query(), containing 'commands'.
    
    Returns:
        A single string with all command outputs concatenated, labeled by command.
    """
    context_parts = []
    commands = classification.get("commands", [])

    # Print routing decision for transparency
    print(f"[Router] Query type : {classification.get('query_type')}")
    print(f"[Router] Reasoning  : {classification.get('reasoning')}")
    print(f"[Router] Executing {len(commands)} command(s):")

    # Divide the context budget evenly across commands
    per_command_limit = MAX_CONTEXT_CHARS // max(len(commands), 1)

    for i, cmd in enumerate(commands, 1):
        print(f"  {i}. {cmd}")
        output = execute_bash(cmd, max_chars=per_command_limit)
        context_parts.append(f"=== Command: {cmd} ===\n{output}")

    return "\n\n".join(context_parts)

## Step 5: Module 4 — Answer Generator

Sends the retrieved context + original question to the LLM to produce a final answer.

In [6]:
def generate_answer(question: str, context: str) -> str:
    """
    Generate a final answer using the LLM, grounded in the retrieved context.
    
    Args:
        question: The original user question.
        context:  The concatenated bash command outputs from retrieve_context().
    
    Returns:
        A detailed answer string with references to specific files.
    """
    prompt = f"""You are an expert code assistant. Answer the user's question using ONLY
the context retrieved from the codebase below.

Requirements:
- Be accurate and specific
- Reference concrete file names and paths
- Quote relevant code snippets where helpful
- If the context is insufficient to fully answer, state which parts are uncertain

=== Retrieved Codebase Context ===
{context}
=== End of Context ===

User question: {question}

Provide a detailed answer:"""

    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=2000
    )

    return response.choices[0].message.content.strip()

## Step 6: Main Pipeline — `answer_question()`

Ties all four modules together into a single end-to-end function.

In [7]:
def answer_question(question: str) -> str:
    """
    Full RAG pipeline: Classify → Retrieve → Generate.
    
    Args:
        question: A natural language question about the mcp-gateway-registry codebase.
    
    Returns:
        A detailed answer string grounded in actual codebase content.
    """
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print('='*60)

    # Step 1: Classify the question → decide which bash commands to run
    classification = classify_query(question)

    # Step 2: Execute bash commands → collect codebase context
    print("\n[Retrieving context...]")
    context = retrieve_context(classification)

    # Step 3: Pass context + question to LLM → generate final answer
    print("\n[Generating answer...]")
    answer = generate_answer(question, context)

    return answer

## Step 7: Run Test Questions

### Questions 1–3 (Simple)
These target dependencies, entry points, and file structure — answered with `cat` and `tree`.

In [8]:
# ── Test Questions 1-3: Simple ─────────────────────────────────────────────────
simple_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)",
]

for question in simple_questions:
    answer = answer_question(question)
    print(f"\n{'='*60}")
    print("[ANSWER]")
    print('='*60)
    print(answer)
    print()


Question: What Python dependencies does this project use?

[Retrieving context...]
[Router] Query type : dependency
[Router] Reasoning  : The user is asking about Python dependencies, so we need to examine the project's configuration files to find the relevant information.
[Router] Executing 1 command(s):
  1. cat ./mcp-gateway-registry/pyproject.toml

[Generating answer...]

[ANSWER]
The project `mcp-registry` uses a variety of Python dependencies, which are specified in the `pyproject.toml` file. 

The main dependencies are listed under the `[project.dependencies]` section:
```toml
dependencies = [
    "aiofiles>=24.1.0",
    "fastapi>=0.115.12",
    "itsdangerous>=2.2.0",
    "jinja2>=3.1.6",
    "mcp>=1.9.3",
    "pydantic>=2.11.3",
    "pydantic-settings>=2.0.0",
    "httpx>=0.27.0",
    "python-dotenv>=1.1.0",
    "python-multipart>=0.0.20",
    "uvicorn[standard]>=0.34.2",
    "faiss-cpu>=1.7.4",
    "sentence-transformers>=3.0.0",
    "websockets>=15.0.1",
    "scikit-learn>=1

### Questions 4–5 (Difficult)
These require searching across multiple Python files — the router will use `grep` + `cat` combinations.

In [9]:
# ── Test Questions 4-5: Difficult ─────────────────────────────────────────────
difficult_questions = [
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
]

for question in difficult_questions:
    answer = answer_question(question)
    print(f"\n{'='*60}")
    print("[ANSWER]")
    print('='*60)
    print(answer)
    print()


Question: How does the authentication flow work, from token validation to user authorization?

[Retrieving context...]
[Router] Query type : multi
[Router] Reasoning  : The question requires a combination of code search and documentation analysis to understand the authentication flow from token validation to user authorization.
[Router] Executing 5 command(s):
  1. grep -r --include='*.py' 'authentication' ./mcp-gateway-registry
  2. grep -r --include='*.md' 'authentication' ./mcp-gateway-registry/docs
  3. cat ./mcp-gateway-registry/docs/authentication.md
  4. cat ./mcp-gateway-registry/src/auth.py
  5. grep -r --include='*.py' 'token validation' ./mcp-gateway-registry/src

[Generating answer...]

[ANSWER]
Based on the provided codebase context, the authentication flow in the MCP Gateway Registry project involves several steps, including token validation and user authorization. However, the exact implementation details are not fully available due to the absence of certain files, such

### Question 6 (Very Hard)
Requires synthesizing code + docs + config files across multiple directories.

In [10]:
# ── Test Question 6: Very Hard ─────────────────────────────────────────────────
hard_question = (
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? "
    "What files would need to be modified and what interfaces must be implemented?"
)

answer = answer_question(hard_question)
print(f"\n{'='*60}")
print("[ANSWER]")
print('='*60)
print(answer)


Question: How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?

[Retrieving context...]
[Router] Query type : multi
[Router] Reasoning  : To add support for a new OAuth provider, we need to search for relevant code and documentation, then inspect the contents of those files.
[Router] Executing 5 command(s):
  1. grep -r --include='*.py' 'oauth' ./mcp-gateway-registry
  2. grep -r --include='*.md' 'oauth' ./mcp-gateway-registry/docs
  3. cat ./mcp-gateway-registry/auth.py
  4. cat ./mcp-gateway-registry/docs/authentication.md
  5. grep -r --include='*.py' 'interface' ./mcp-gateway-registry/auth

[Generating answer...]

[ANSWER]
To add support for a new OAuth provider, such as Okta, to the authentication system, you would need to modify the following files and implement the required interfaces:

1. **`credentials-provider/oauth/oauth_providers.yaml`**: This file contain